<h1 align="center">
Visualization Tools for Bounding Boxes, General Captioning and Pointing
</h1>

This `.ipynb` file is intended for visualizing the output samples of the Bounding Boxes annotation algorithm, General Captions, and Pointing data, as well as some statistics derived from them.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from PIL import Image

root = Path("..").resolve()

## Sanity-check functions

In [ ]:
def _check_json(json_path: Path | str) -> None:
    """Checks that a JSON file exists and can be successfully parsed."""

    path = Path(json_path)

    if not path.is_file():
        raise FileNotFoundError(f"JSON file not found: {path}")

    try:
        with path.open("r", encoding="utf-8") as f:
            json.load(f)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON format in file: {path}") from exc

## Scale functions

In [ ]:
def _axis_scale(ax: plt.Axes, image_w: int, image_h: int) -> tuple[float, float]:
    """Returns pixels-per-image-pixel scale factors for the current axis."""
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bbox = ax.get_window_extent(renderer=renderer)
    return bbox.width / image_w, bbox.height / image_h


def _style_for_axis(
    ax: plt.Axes, image_w: int, image_h: int
) -> tuple[float, float, float]:
    """Derives line width, point radius and font size for readable overlays."""
    sx, sy = _axis_scale(ax, image_w, image_h)
    s = (sx + sy) / 2.0  # screen-px per image-px
    # Эти коэффициенты дают “похожий” вид на разных масштабах.
    line_w = max(2.0, min(6.0, 3.0 / max(s, 1e-6)))
    point_r = max(5.0, min(12.0, 7.0 / max(s, 1e-6)))
    font_sz = max(10.0, min(18.0, 12.0 / max(s, 1e-6)))
    return line_w, point_r, font_sz

## Bounding Boxes Visualization

In [ ]:
def show_bboxes(json_path: Path | str) -> None:
    """Shows an image with labeled bounding boxes and the question above it."""

    json_path = Path(root / json_path)

    _check_json(json_path)

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    images = data.get("images")
    question = str(data.get("question", ""))
    answer = data.get("answer") or []

    image_path = Path(images[0])
    if not image_path.is_absolute():
        image_path = root / image_path

    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    fig = plt.figure(figsize=(12, 8))
    top_text_h = 0.12
    left, right, bottom = 0.02, 0.98, 0.02

    ax = fig.add_axes([left, bottom, right - left, 1.0 - bottom - top_text_h])
    ax.imshow(image)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis("off")

    line_w, _, font_sz = _style_for_axis(ax, w, h)

    for item in answer:
        if not isinstance(item, dict):
            continue

        bbox = item.get("bbox_2d")
        label = item.get("label")

        if (
            not isinstance(bbox, list)
            or len(bbox) != 4
            or not all(isinstance(v, (int, float)) for v in bbox)
            or not isinstance(label, str)
        ):
            continue

        x1, y1, x2, y2 = (
            float(bbox[0]),
            float(bbox[1]),
            float(bbox[2]),
            float(bbox[3]),
        )
        if x2 <= x1 or y2 <= y1:
            continue

        rect = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=line_w,
            edgecolor="lime",
        )
        ax.add_patch(rect)

        text = label.strip()
        if text:
            ax.text(
                x1,
                y1,
                text,
                fontsize=font_sz,
                fontweight="bold",
                color="white",
                va="bottom",
                ha="left",
                bbox=dict(facecolor="black", edgecolor="none", alpha=0.8, pad=2),
            )

    safe_q = question.replace("$", r"\$")
    fig.text(
        left, 0.98, rf"$\bf{{Question}}$ {safe_q}", va="top", ha="left", fontsize=12
    )

    plt.show()

## Pointing Visualization

In [ ]:
def show_points(json_path: Path | str) -> None:
    """Shows an image with labeled points and the question above it."""

    json_path = Path(root / json_path)

    _check_json(json_path)

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    images = data.get("images")
    question = str(data.get("question", ""))
    answer = data.get("answer") or []

    image_path = Path(images[0])
    if not image_path.is_absolute():
        image_path = root / image_path

    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    fig = plt.figure(figsize=(12, 8))
    top_text_h = 0.12
    left, right, bottom = 0.02, 0.98, 0.02

    ax = fig.add_axes([left, bottom, right - left, 1.0 - bottom - top_text_h])
    ax.imshow(image)
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis("off")

    line_w, point_r, font_sz = _style_for_axis(ax, w, h)

    for item in answer:
        if not isinstance(item, dict):
            continue

        point = item.get("point_2d")
        label = item.get("label")

        if (
            not isinstance(point, list)
            or len(point) != 2
            or not all(isinstance(v, (int, float)) for v in point)
            or not isinstance(label, str)
        ):
            continue

        x, y = float(point[0]), float(point[1])
        text = label.strip()
        if not text:
            continue

        circ = Circle(
            (x, y),
            radius=point_r,
            facecolor="lime",
            edgecolor="black",
            linewidth=line_w,
        )
        ax.add_patch(circ)

        ax.text(
            x + point_r + 2,
            y,
            text,
            fontsize=font_sz,
            fontweight="bold",
            color="white",
            va="center",
            ha="left",
            bbox=dict(facecolor="black", edgecolor="none", alpha=0.8, pad=2),
        )

    safe_q = question.replace("$", r"\$")
    fig.text(
        left, 0.98, rf"$\bf{{Question}}$ {safe_q}", va="top", ha="left", fontsize=12
    )

    plt.show()

## General Captioning Visualization

In [ ]:
def _wrap_text_to_pixel_width(
    fig: plt.Figure,
    text: str,
    max_width_px: float,
    *,
    fontsize: int,
    max_lines: int,
) -> str:
    """Wraps text by words to fit a given pixel width, truncating with ellipsis if needed."""

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    def measure(s: str) -> float:
        t = fig.text(0, 0, s, fontsize=fontsize, alpha=0.0)
        fig.canvas.draw()
        w = t.get_window_extent(renderer=renderer).width
        t.remove()
        return w

    words = " ".join(text.split()).split(" ")
    if not words:
        return ""

    lines: list[str] = []
    current: list[str] = []

    for w in words:
        candidate = " ".join(current + [w])
        if current and measure(candidate) > max_width_px:
            lines.append(" ".join(current))
            current = [w]
            if len(lines) >= max_lines:
                break
        else:
            current.append(w)

    if len(lines) < max_lines and current:
        lines.append(" ".join(current))

    full = " ".join(words).strip()
    joined = " ".join(lines).strip()
    if joined == full and len(lines) <= max_lines:
        return "\n".join(lines)

    lines = lines[:max_lines] if lines else [""]
    last = lines[-1]
    ellipsis = "..."

    if not last:
        lines[-1] = ellipsis
        return "\n".join(lines)

    while last and measure(last + ellipsis) > max_width_px:
        last = last.rsplit(" ", 1)[0] if " " in last else last[:-1]

    lines[-1] = (last + ellipsis).strip()
    return "\n".join(lines)


def show_captioning(json_path: Path | str) -> None:
    """Visualizes an image with wrapped question and answer text above it."""

    json_path = Path(root / json_path)

    _check_json(json_path)

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    images = data.get("images")
    question = str(data.get("question", ""))
    answer = str(data.get("answer", ""))

    image_path = Path(images[0])
    if not image_path.is_absolute():
        image_path = root / image_path

    image = Image.open(image_path).convert("RGB")

    fig = plt.figure(figsize=(12, 8))

    # Reserve the top for the text: 24% of the figure height.
    top_text_h = 0.24
    left = 0.02
    right = 0.98
    bottom = 0.02

    ax = fig.add_axes([left, bottom, right - left, 1.0 - bottom - top_text_h])
    ax.imshow(image)
    ax.axis("off")

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    ax_bbox = ax.get_window_extent(renderer=renderer)
    max_width_px = ax_bbox.width

    safe_question = question.replace("$", r"\$")
    safe_answer = answer.replace("$", r"\$")

    q_text = _wrap_text_to_pixel_width(
        fig, safe_question, max_width_px, fontsize=12, max_lines=2
    )
    a_text = _wrap_text_to_pixel_width(
        fig, safe_answer, max_width_px, fontsize=12, max_lines=5
    )

    y_top = 1.0 - 0.02
    fig.text(
        left, y_top, rf"$\bf{{Question}}$ {q_text}", va="top", ha="left", fontsize=12
    )
    fig.text(
        left,
        y_top - 0.08,
        rf"$\bf{{Answer}}$ {a_text}",
        va="top",
        ha="left",
        fontsize=12,
    )

    plt.show()

In [ ]:
show_bboxes(json_path="outputs/bboxing/00000009.json")

In [ ]:
show_points(json_path="outputs/pointing/00000009.json")

In [ ]:
show_captioning(json_path="outputs/captions/00000009.json")